# Translation-fidelity check — Gemini on ORIGINAL vs TRIPADVISOR-ENGLISH text (133 reviews)

Run the cells from top to bottom. If anything stops, run the same cell again: it continues where it stopped.

| Part | What | Reviews |
|---|---|---|
| TEST | 1 tiny request | 2 |
| EN | English translation (the text analysed in the study), **full text** | 133 |
| ORIG | original-language text (FR/ES/DE/IT), **full text** | 133 |
| EN2 | English again (measures run-to-run noise) | 133 |

Needs in the same folder: `LLM_FULL_ANALYSIS.csv` and `Translation_Check_Collection_FILLED.xlsx`.
The outputs are written to `translation_results.zip`. Expected cost ≈ 0.5–0.8 USD, ≈ 10 minutes.

In [ ]:
# ============ CELL 1 : SETTINGS ============
RUN_EN, RUN_ORIG, RUN_EN2 = True, True, True
MODEL_NAME  = 'gemini-2.5-flash'
BATCH_SIZE  = 15
MAX_WORKERS = 4
MAX_TRIES   = 3
TRUNC       = 500          # not used here (full text), kept for the engine
PRICE_INPUT, PRICE_OUTPUT = 0.30, 2.50
CORPUS_FILE = 'LLM_FULL_ANALYSIS.csv'
SAMPLE_FILE = 'Translation_Check_Collection_FILLED.xlsx'
OUT_DIR     = 'translation_outputs'

In [ ]:
# ============ CELL 2 : IMPORTS + API KEY (typed, never saved) ============
import os, sys, json, time, platform, datetime, zipfile, getpass
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import google.generativeai as genai
from tqdm import tqdm

API_KEY = getpass.getpass('Paste your Gemini API key and press Enter: ')
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel(MODEL_NAME)
os.makedirs(OUT_DIR, exist_ok=True)
print('OK - library version:', genai.__version__, '| model:', MODEL_NAME)

In [ ]:
# ============ CELL 3 : LOAD + CHECKS ============
corpus = pd.read_csv(CORPUS_FILE)
row_of = {str(t).strip(): i for i, t in enumerate(corpus['text'])}
s = pd.read_excel(SAMPLE_FILE, sheet_name='Collection')
s = s[s['original_text'].notna() & s['tripadvisor_english_translation'].notna()].copy()
s['en'] = s['tripadvisor_english_translation'].astype(str).str.strip()
s['orig'] = s['original_text'].astype(str).str.strip()
s['row_id'] = s['en'].map(row_of)
assert len(s) == 133, f'Expected 133 reviews, found {len(s)}'
assert s['row_id'].notna().all(), 'Some English texts are not in LLM_FULL_ANALYSIS.csv - check the input files'
assert s['row_id'].is_unique, 'Duplicate reviews - check the input files'
s['row_id'] = s['row_id'].astype(int)
s = s.sort_values('row_id')                      # same order (same batches) for EN and ORIG
EN   = dict(zip(s['row_id'], s['en']))
ORIG = dict(zip(s['row_id'], s['orig']))
s[['row_id', 'site', 'language', 'stars', 'review_link']].to_csv(os.path.join(OUT_DIR, 'sample_index.csv'), index=False)
print('Check OK: 133 reviews, all found in the corpus.')
print(s['language'].value_counts().to_dict())

In [ ]:
# ============ CELL 4 : PROMPT (original, word for word) ============
# ---- (a) ORIGINAL prompt, copied word for word from the original notebook ----
def original_prompt(reviews_json):
    return """You are a tourism analyst. Analyze each TripAdvisor review and return ONLY a JSON array (same length and order as input). No explanation, no markdown.

For each review, return:
{
  "overall_sentiment": "Positive" | "Negative" | "Neutral",
  "sentiment_score": 1-5,
  "emotional_tone": "Joy" | "Anger" | "Fear" | "Disgust" | "Sadness" | "Surprise" | "Trust" | "Frustration" | "Neutral" | "Other",
  "aspects": {
    "safety": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "pricing": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "service": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "cleanliness": "Positive" | "Negative" | "Neutral" | "Not_mentioned",
    "atmosphere": "Positive" | "Negative" | "Neutral" | "Not_mentioned"
  },
  "problems": ["Aggressive_vendors" | "Navigation_difficulty" | "Getting_lost" | "Bad_smell" | "Overcrowding" | "Poor_condition" | "High_prices" | "Poor_cleanliness" | "Scam" | "Safety" | "Other"],
  "economic_signals": {
    "price_perception": "Expensive" | "Fair" | "Cheap" | "Not_mentioned",
    "recommendation": "Yes" | "No" | "Not_mentioned",
    "revisit_intention": "Yes" | "No" | "Not_mentioned"
  },
  "keywords": ["keyword1", "keyword2", "keyword3"]
}

Rules:
- If no problems, return "problems": []
- Maximum 3 problems per review
- Maximum 3-5 keywords per review
- Translate any non-English problems/keywords into English
- Be consistent: same issue = same wording across all reviews
- Return ONLY the JSON array, nothing else

Examples:
"Beautiful place but shopkeepers are aggressive and everything is overpriced"
→ {"overall_sentiment": "Negative", "sentiment_score": 2, "emotional_tone": "Frustration", "aspects": {"safety": "Not_mentioned", "pricing": "Negative", "service": "Negative", "cleanliness": "Not_mentioned", "atmosphere": "Positive"}, "problems": ["Aggressive_vendors", "High_prices"], "economic_signals": {"price_perception": "Expensive", "recommendation": "Not_mentioned", "revisit_intention": "Not_mentioned"}, "keywords": ["beautiful", "aggressive", "overpriced"]}

"Amazing experience, felt very safe, would definitely come back!"
→ {"overall_sentiment": "Positive", "sentiment_score": 5, "emotional_tone": "Joy", "aspects": {"safety": "Positive", "pricing": "Not_mentioned", "service": "Not_mentioned", "cleanliness": "Not_mentioned", "atmosphere": "Positive"}, "problems": [], "economic_signals": {"price_perception": "Not_mentioned", "recommendation": "Yes", "revisit_intention": "Yes"}, "keywords": ["amazing", "safe", "come back"]}

Reviews:
""" + reviews_json + """

Return ONLY the JSON array."""

In [ ]:
# ============ CELL 5 : ENGINE (same logic as the original + token logging) ============
LOG_FILE = os.path.join(OUT_DIR, 'token_log.csv')

def usage_of(resp):
    u = getattr(resp, 'usage_metadata', None)
    p = getattr(u, 'prompt_token_count', 0) or 0
    c = getattr(u, 'candidates_token_count', 0) or 0
    t = getattr(u, 'total_token_count', 0) or 0
    th = getattr(u, 'thoughts_token_count', None)
    if th is None:
        th = max(0, t - p - c)
    return p, c, th, t

def call_batch(part, batch_id, texts, prompt_fn):
    reviews_json = json.dumps(texts, ensure_ascii=False)
    prompt = prompt_fn(reviews_json)
    logs = []
    for attempt in range(1, MAX_TRIES + 1):
        t0 = time.time()
        rec = {'part': part, 'batch_id': batch_id, 'attempt': attempt, 'n_reviews': len(texts),
               'input_chars': len(reviews_json), 'prompt_tokens': 0, 'output_tokens': 0,
               'thinking_tokens': 0, 'total_tokens': 0, 'seconds': 0, 'ok': False, 'error': '',
               'model_version': '', 'timestamp': datetime.datetime.now().isoformat(timespec='seconds')}
        try:
            resp = model.generate_content(prompt, request_options={'timeout': 180})
            p, c, th, t = usage_of(resp)
            rec.update(prompt_tokens=p, output_tokens=c, thinking_tokens=th, total_tokens=t,
                       model_version=str(getattr(resp, 'model_version', '') or ''))
            content = resp.text.strip().replace('```json', '').replace('```', '').strip()
            data = json.loads(content)
            if not isinstance(data, list) or len(data) != len(texts):
                raise ValueError(f'Length mismatch: expected {len(texts)} got {len(data) if isinstance(data, list) else "?"}')
            rec.update(ok=True, seconds=round(time.time() - t0, 2)); logs.append(rec)
            return data, logs
        except Exception as e:
            rec.update(error=str(e)[:150], seconds=round(time.time() - t0, 2)); logs.append(rec)
            time.sleep(3)
    return None, logs

def run_part(part, texts_by_id, prompt_fn):
    row_ids = list(texts_by_id.keys())
    out_file = os.path.join(OUT_DIR, f'{part}.csv')
    done = set()
    if os.path.exists(out_file):
        done = set(pd.read_csv(out_file)['row_id'].tolist())
    todo = [i for i in row_ids if i not in done]
    print(f'\n=== {part}: {len(row_ids)} reviews | already done {len(done)} | to do {len(todo)} ===')
    if not todo:
        return
    batches = [todo[b:b + BATCH_SIZE] for b in range(0, len(todo), BATCH_SIZE)]
    bar = tqdm(total=len(batches), desc=part) if tqdm else None
    failed = 0
    for start in range(0, len(batches), MAX_WORKERS):
        chunk = batches[start:start + MAX_WORKERS]
        rows, logs = [], []
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = {}
            for j, b in enumerate(chunk):
                texts = [texts_by_id[i] for i in b]
                futs[ex.submit(call_batch, part, f'{start + j}', texts, prompt_fn)] = b
            for f in as_completed(futs):
                b = futs[f]
                data, lg = f.result()
                logs.extend(lg)
                if data is None:
                    failed += len(b)
                else:
                    for i, res in zip(b, data):
                        rows.append({'row_id': i, 'raw_json': json.dumps(res, ensure_ascii=False)})
                if bar: bar.update(1)
        # save after every chunk of 4 batches
        if rows:
            pd.DataFrame(rows).to_csv(out_file, mode='a', header=not os.path.exists(out_file), index=False)
        pd.DataFrame(logs).to_csv(LOG_FILE, mode='a', header=not os.path.exists(LOG_FILE), index=False)
        time.sleep(1)
    if bar: bar.close()
    n_done = len(pd.read_csv(out_file)) if os.path.exists(out_file) else 0
    print(f'{part}: finished {n_done}/{len(row_ids)}' + (f' | {failed} failed -> run this cell again to retry them' if failed else ''))

In [ ]:
# ============ CELL 6 : TEST (2 reviews) ============
ids = list(ORIG)[:2]
data, lg = call_batch('TEST', 'test', [ORIG[i] for i in ids], original_prompt)
print('Result OK' if data else 'FAILED: ' + lg[-1]['error'])
if data: print(json.dumps(data[0], ensure_ascii=False)[:300])

In [ ]:
# ============ CELL 7 : RUN EN, ORIG, EN2 (about 10 minutes) ============
if RUN_EN:   run_part('EN',   EN,   original_prompt)
if RUN_ORIG: run_part('ORIG', ORIG, original_prompt)
if RUN_EN2:  run_part('EN2',  EN,   original_prompt)

In [ ]:
# ============ CELL 8 : SUMMARY + ZIP ============
log = pd.read_csv(LOG_FILE); log = log[log.part != 'TEST']
rows = []
for part in ['EN', 'ORIG', 'EN2']:
    f = os.path.join(OUT_DIR, f'{part}.csv')
    n = len(pd.read_csv(f)) if os.path.exists(f) else 0
    g = log[log.part == part]
    cost = g.prompt_tokens.sum()/1e6*PRICE_INPUT + (g.output_tokens.sum()+g.thinking_tokens.sum())/1e6*PRICE_OUTPUT
    rows.append({'part': part, 'done': f'{n}/133', 'requests': len(g), 'failed_attempts': int((~g.ok).sum()), 'cost_usd': round(cost, 3)})
summary = pd.DataFrame(rows); print(summary.to_string(index=False)); print('TOTAL cost (USD):', round(summary.cost_usd.sum(), 2))
json.dump({'run_date': datetime.datetime.now().isoformat(timespec='seconds'), 'model': MODEL_NAME,
           'model_versions_seen': sorted(set(log.model_version.dropna().astype(str))),
           'library': 'google-generativeai ' + genai.__version__, 'batch_size': BATCH_SIZE, 'workers': MAX_WORKERS,
           'input': 'full text (no truncation)', 'generation_config': 'default (same as original run)'},
          open(os.path.join(OUT_DIR, 'run_info.json'), 'w'), indent=2)
summary.to_csv(os.path.join(OUT_DIR, 'summary.csv'), index=False)
with zipfile.ZipFile('translation_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(OUT_DIR): z.write(os.path.join(OUT_DIR, fn), fn)
print('DONE -> translation_results.zip (no API key inside).')